# Loan application screening model

This educational notebook learns to reproduce the source CSV's historical **Approved/Rejected** labels. The dataset has no repayment outcomes, so it cannot train a default model. It is not evidence for real credit decisions. Money columns stay in the dataset's original, unverified units; no currency conversion is assumed.

Run the single code cell to validate the data, compare models using cross-validation on the training split, evaluate once on a held-out split, and save the model and a small model card.

In [ ]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score, brier_score_loss
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RAW_DATA = Path("data/raw/loan_approval_dataset.csv")
MODEL_DIR = Path("models")
PROCESSED_DIR = Path("data/processed")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# This source contains past approval/rejection labels, not repayment outcomes.
raw = pd.read_csv(RAW_DATA)
raw.columns = raw.columns.str.strip()
required = {"no_of_dependents", "education", "self_employed", "income_annum", "loan_amount", "loan_term", "cibil_score", "residential_assets_value", "commercial_assets_value", "luxury_assets_value", "bank_asset_value", "loan_status"}
missing = required - set(raw.columns)
if missing:
    raise ValueError(f"Dataset is missing required columns: {sorted(missing)}")

# Preserve source monetary units. No undocumented currency conversion is applied.
df = raw.copy()
for col in ["education", "self_employed", "loan_status"]:
    df[col] = df[col].astype(str).str.strip()
df["is_self_employed"] = df["self_employed"].map({"Yes": 1, "No": 0})
df["is_graduate"] = df["education"].map({"Graduate": 1, "Not Graduate": 0})
df["rejected"] = df["loan_status"].map({"Approved": 0, "Rejected": 1})
df["total_assets"] = df[["residential_assets_value", "commercial_assets_value", "luxury_assets_value", "bank_asset_value"]].clip(lower=0).sum(axis=1)
df["debt_to_income"] = df["loan_amount"] / df["income_annum"].replace(0, np.nan)
df["asset_coverage"] = df["total_assets"] / df["loan_amount"].replace(0, np.nan)
df = df.rename(columns={"income_annum": "income", "loan_term": "loan_term", "cibil_score": "credit_score", "no_of_dependents": "dependents"})
FEATURES = ["credit_score", "debt_to_income", "asset_coverage", "loan_term", "dependents", "is_self_employed", "is_graduate"]
df = df[FEATURES + ["rejected"]].replace([np.inf, -np.inf], np.nan).dropna()
if df["rejected"].nunique() != 2 or df["rejected"].value_counts().min() < 5:
    raise ValueError("Need at least five examples from each approval/rejection class.")
print(f"Rows: {len(df)} | approval/rejection counts: {df['rejected'].value_counts().to_dict()}")
print("Target is historical rejection, not borrower default.")

X, y = df[FEATURES], df["rejected"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
models = {
    "Logistic Regression": Pipeline([("scale", StandardScaler()), ("model", LogisticRegression(class_weight="balanced", max_iter=2000, random_state=42))]),
    "Random Forest": RandomForestClassifier(n_estimators=300, class_weight="balanced", min_samples_leaf=3, random_state=42, n_jobs=-1),
}
# XGBoost is optional; never install packages from inside a notebook cell.
try:
    from xgboost import XGBClassifier
    models["XGBoost"] = XGBClassifier(n_estimators=250, max_depth=3, learning_rate=0.05, subsample=0.9, colsample_bytree=0.9, eval_metric="logloss", random_state=42, n_jobs=-1)
except ImportError:
    print("XGBoost unavailable; comparing the installed scikit-learn models.")

scores = {}
for name, model in models.items():
    fold_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=None)
    scores[name] = float(fold_scores.mean())
    print(f"{name}: CV ROC AUC {fold_scores.mean():.3f} +/- {fold_scores.std():.3f}")
best_name = max(scores, key=scores.get)
model = CalibratedClassifierCV(estimator=models[best_name], method="sigmoid", cv=cv)
model.fit(X_train, y_train)
prob = model.predict_proba(X_test)[:, list(model.classes_).index(1)]
pred = (prob >= 0.5).astype(int)
print(f"\nSelected by training cross-validation: {best_name}")
print("Held-out report (0=Approved, 1=Rejected):")
print(classification_report(y_test, pred, target_names=["Approved", "Rejected"], zero_division=0))
print("Confusion matrix:")
print(confusion_matrix(y_test, pred))
print(f"Held-out ROC AUC: {roc_auc_score(y_test, prob):.3f}")
print(f"Held-out average precision: {average_precision_score(y_test, prob):.3f}")
print(f"Held-out Brier score (lower is better): {brier_score_loss(y_test, prob):.3f}")
print("These metrics measure reproduction of historical decisions only. They do not establish default prediction, causal explanations, fairness, or future performance.")

joblib.dump(model, MODEL_DIR / "loan_model.pkl")
(PROCESSED_DIR / "model_card.json").write_text(json.dumps({
    "target": "historical loan_status rejection (not repayment default)",
    "positive_class": 1,
    "selected_model": best_name,
    "probability_calibration": "sigmoid, stratified 5-fold cross-validation on training split",
    "selection_metric": "mean 5-fold stratified training ROC AUC",
    "cv_roc_auc": scores,
    "test_roc_auc": float(roc_auc_score(y_test, prob)),
    "test_average_precision": float(average_precision_score(y_test, prob)),
    "test_brier_score": float(brier_score_loss(y_test, prob)),
    "test_rows": int(len(y_test)),
    "source_currency": "unverified; source dataset units retained",
}, indent=2), encoding="utf-8")
print(f"Saved model to {MODEL_DIR / 'loan_model.pkl'} and evaluation summary to {PROCESSED_DIR / 'model_card.json'}")
